# Text Classification Analysis

This notebook provides a deep analysis of the text classification results produced by the pipeline. It evaluates the semantic structure of categories and identifies factors contributing to classification performance.

## Hypotheses Analyzed:
1. **Semantic Projection**: Low-dimensional projection reveals the semantic structure and potential overlaps between categories.
2. **Centrality and Difficulty**: Categories that are semantically "central" (closer to the global mean) are more difficult to classify.
3. **Error Closeness**: Semantically closer categories are more likely to be confused with each other.
4. **Semantic Volume**: Categories with higher semantic dispersion among their texts are harder to classify.

## 1. Setup and Data Loading
Initialize libraries and load outputs from the classification pipeline.

In [ ]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn scipy sentence-transformers

import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score
from sklearn.manifold import TSNE
from scipy.spatial.distance import cosine
from scipy.stats import pearsonr
from sentence_transformers import SentenceTransformer

# Configuration
CONFIG = {
    'OUTPUT_DIR': '/content/drive/MyDrive/classification_outputs',
    'EMBEDDING_MODEL_NAME': 'all-MiniLM-L6-v2'
}

embedding_model = SentenceTransformer(CONFIG['EMBEDDING_MODEL_NAME'])

def load_cache(name):
    path = os.path.join(CONFIG['OUTPUT_DIR'], f'{name}_latest.json')
    if os.path.exists(path):
        with open(path, 'r') as f:
            return json.load(f)
    return None

test_data = load_cache('test_data')
descriptions_31 = load_cache('descriptions_31')
results_31 = load_cache('classification_results_31')
results_35 = load_cache('classification_results_35')
ablation_descriptions = load_cache('ablation_descriptions')

df_results = pd.DataFrame(results_31)
categories = sorted(df_results['label'].unique().tolist())
print(f'Loaded results for {len(categories)} categories.')

## 2. Embedding Utilities
Helper functions for computing and caching embeddings.

In [ ]:
embedding_cache = load_cache('embedding_cache') or {}

def get_embedding(text):
    if text in embedding_cache:
        return np.array(embedding_cache[text])
    embedding = embedding_model.encode([text])[0]
    embedding_cache[text] = embedding.tolist()
    return embedding

def save_embedding_cache():
    path = os.path.join(CONFIG['OUTPUT_DIR'], 'embedding_cache_latest.json')
    with open(path, 'w') as f:
        json.dump(embedding_cache, f)


## 3. Primary Performance Evaluation
Global metrics and confusion matrix for the main classification run (Gemini 3.1 Flash lite).

In [ ]:
def evaluate_performance(df, title_suffix=''):
    accuracy = accuracy_score(df['label'], df['prediction'])
    print(f'Global Accuracy {title_suffix}: {accuracy:.4f}')

    report = classification_report(df['label'], df['prediction'])
    print(report)

    plt.figure(figsize=(10, 8))
    cm = confusion_matrix(df['label'], df['prediction'], labels=categories)
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=categories, yticklabels=categories)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(f'Confusion Matrix {title_suffix}')
    plt.show()
    return accuracy

evaluate_performance(df_results, '(Gemini 3.1 Flash lite)')

## 3. Hypothesis 1: Semantic Projection
**Hypothesis**: Low-dimensional projection of category centroids (from texts) and description embeddings reveals semantic overlaps that might lead to misclassification.

**Methodology**: Calculate text centroids for each category and embeddings for each description. Use t-SNE to project these into 2D space.

In [ ]:
category_centroids = {}
for cat in categories:
    texts = test_data[cat]
    embeddings = [get_embedding(t) for t in texts]
    category_centroids[cat] = np.mean(embeddings, axis=0)

desc_embeddings = {cat: get_embedding(desc) for cat, desc in descriptions_31.items() if cat in categories}
save_embedding_cache()

# Prepare data for t-SNE
all_vectors = []
labels = []
types = []

for cat in categories:
    all_vectors.append(category_centroids[cat])
    labels.append(cat)
    types.append('Text Centroid')
    
    if cat in desc_embeddings:
        all_vectors.append(desc_embeddings[cat])
        labels.append(cat)
        types.append('Description')

tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(all_vectors)-1))
coords = tsne.fit_transform(np.array(all_vectors))

df_coords = pd.DataFrame({
    'x': coords[:, 0],
    'y': coords[:, 1],
    'label': labels,
    'type': types
})

plt.figure(figsize=(12, 10))
sns.scatterplot(data=df_coords, x='x', y='y', hue='label', style='type', s=100)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.title('Low-dimensional Projection of Category Embeddings')
plt.show()

## 4. Error Analysis
Detailed breakdown of misclassified texts per category.

In [ ]:
errors = df_results[df_results['label'] != df_results['prediction']]
print(f'Total errors: {len(errors)} / {len(df_results)}')

for cat in categories:
    cat_errors = errors[errors['label'] == cat]
    if not cat_errors.empty:
        print(f'\nCategory: {cat}')
        for _, row in cat_errors.iterrows():
            print(f"  - Predicted: {row['prediction']} | Text: {row['text']}")

## 5. Hypothesis 2: Centrality and Difficulty
**Hypothesis**: Categories that are semantically "central" (closer to all other categories) are more difficult to classify correctly.

**Methodology**: Calculate category centrality as the average cosine distance from its centroid to all other category centroids. Correlate this with the category's F1-score.

In [ ]:
dist_matrix = np.zeros((len(categories), len(categories)))
for i, cat1 in enumerate(categories):
    for j, cat2 in enumerate(categories):
        dist_matrix[i, j] = cosine(category_centroids[cat1], category_centroids[cat2])

centrality = dist_matrix.mean(axis=1)

report = classification_report(df_results['label'], df_results['prediction'], output_dict=True)
cat_f1 = [report[cat]['f1-score'] for cat in categories]

df_centrality = pd.DataFrame({
    'Category': categories,
    'Centrality (Avg Distance)': centrality,
    'F1-Score': cat_f1
})

plt.figure(figsize=(8, 6))
sns.regplot(data=df_centrality, x='Centrality (Avg Distance)', y='F1-Score')
plt.title('Centrality vs Classification Performance')
plt.show()

corr, p = pearsonr(df_centrality['Centrality (Avg Distance)'], df_centrality['F1-Score'])
print(f'Pearson Correlation: {corr:.4f} (p={p:.4f})')

## 6. Hypothesis 3: Error Closeness and Semantic Distance
**Hypothesis**: Categories that are semantically closer are more difficult to misrepresent (i.e., more likely to be confused with each other).

**Methodology**: Evaluate the correlation between the semantic distance matrix and the confusion matrix (off-diagonal elements).

In [ ]:
cm = confusion_matrix(df_results['label'], df_results['prediction'], labels=categories)

# Flatten both matrices and exclude diagonal (self-classification)
indices = ~np.eye(len(categories), dtype=bool)
flat_dist = dist_matrix[indices]
flat_errors = cm[indices]

plt.figure(figsize=(8, 6))
sns.regplot(x=flat_dist, y=flat_errors)
plt.xlabel('Semantic Distance')
plt.ylabel('Confusion Count')
plt.title('Semantic Distance vs Category Confusion')
plt.show()

corr, p = pearsonr(flat_dist, flat_errors)
print(f'Pearson Correlation: {corr:.4f} (p={p:.4f})')
print('A negative correlation suggests closer categories are confused more often.')

## 7. Hypothesis 4: Semantic Volume and Dispersion
**Hypothesis**: Categories with higher semantic dispersion (texts are spread out semantically) are harder to classify accurately.

**Methodology**: Calculate semantic dispersion as the average cosine distance of a category's texts to its centroid. Correlate this with classification accuracy per category.

In [ ]:
dispersions = []
for cat in categories:
    texts = test_data[cat]
    centroid = category_centroids[cat]
    dists = [cosine(get_embedding(t), centroid) for t in texts]
    dispersions.append(np.mean(dists))

cat_acc = [report[cat]['recall'] for cat in categories] # Recall is equivalent to accuracy per class

df_dispersion = pd.DataFrame({
    'Category': categories,
    'Dispersion': dispersions,
    'Accuracy (Recall)': cat_acc
})

plt.figure(figsize=(8, 6))
sns.regplot(data=df_dispersion, x='Dispersion', y='Accuracy (Recall)')
plt.title('Semantic Dispersion vs Category Accuracy')
plt.show()

corr, p = pearsonr(df_dispersion['Dispersion'], df_dispersion['Accuracy (Recall)'])
print(f'Pearson Correlation: {corr:.4f} (p={p:.4f})')

## 8. Semantic Distance Preservation Metric
Evaluate how distances across categories were preserved in descriptions compared to raw texts.

In [ ]:
dist_desc = np.zeros((len(categories), len(categories)))
for i, cat1 in enumerate(categories):
    for j, cat2 in enumerate(categories):
        dist_desc[i, j] = cosine(desc_embeddings[cat1], desc_embeddings[cat2])

raw_flat = dist_matrix[np.triu_indices(len(categories), k=1)]
desc_flat = dist_desc[np.triu_indices(len(categories), k=1)]
preservation_score, _ = pearsonr(raw_flat, desc_flat)

print(f'Semantic Distance Preservation Score: {preservation_score:.4f}')

## 9. Preservation vs Accuracy Correlation
Determine if better preservation of semantic distance correlates with higher accuracy.

In [ ]:
cat_preservation = {}
for i, cat in enumerate(categories):
    cat_preservation[cat] = pearsonr(dist_matrix[i], dist_desc[i])[0]

df_corr = pd.DataFrame({
    'Accuracy': [report[cat]['recall'] for cat in categories],
    'Preservation': pd.Series(cat_preservation)
})

plt.figure(figsize=(8, 6))
sns.regplot(data=df_corr, x='Preservation', y='Accuracy')
plt.title('Semantic Preservation vs Accuracy')
plt.show()

corr_val, p_val = pearsonr(df_corr['Preservation'], df_corr['Accuracy'])
print(f'Pearson Correlation: {corr_val:.4f} (p={p_val:.4f})')

## 10. Ablation Study Analysis
Compare results with descriptions generated by Gemini 3.5 Flash in a single request.

In [ ]:
if results_35:
    df_results_35 = pd.DataFrame(results_35)
    evaluate_performance(df_results_35, '(Gemini 3.5 Flash - Ablation)')
    
    # Calculate preservation for ablation
    dist_desc_35 = np.zeros((len(categories), len(categories)))
    desc_embeddings_35 = {cat: get_embedding(desc) for cat, desc in ablation_descriptions.items() if cat in categories}
    
    for i, cat1 in enumerate(categories):
        for j, cat2 in enumerate(categories):
            dist_desc_35[i, j] = cosine(desc_embeddings_35[cat1], desc_embeddings_35[cat2])
            
    desc_flat_35 = dist_desc_35[np.triu_indices(len(categories), k=1)]
    pres_score_35, _ = pearsonr(raw_flat, desc_flat_35)
    print(f'Ablation Semantic Distance Preservation Score: {pres_score_35:.4f}')
else:
    print('No ablation results found.')